# Experiment: MC Dropout

**Loss:** BCE (unified for fair comparison)
**Seeds:** 42, 123, 456 (3 runs)

In [ ]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import gc, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import DistMult
from src.models.uncertain_kge import MCDropoutKGE
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

In [ ]:
def train_and_evaluate(seed):
    print(f"\n{'='*50}")
    print(f"SEED {seed}")
    print(f"{'='*50}")
    set_seed(seed)
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Training
    base = DistMult(train_data.num_entities, train_data.num_relations, embedding_dim=200, dropout=0.3)
    model = MCDropoutKGE(base, num_samples=20).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)

    for ep in (pbar := tqdm(range(50), desc=f"MCDropout (seed={seed})")):
        model.train()
        loss_sum, n = 0, 0
        for st in range(0, len(train_data), 1024):
            pos = torch.tensor(train_data.triples[st:st+1024], device=device)
            neg = pos.clone()
            neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
            opt.zero_grad()
            pos_s = model.base_model(pos[:,0], pos[:,1], pos[:,2])
            neg_s = model.base_model(neg[:,0], neg[:,1], neg[:,2])
            loss = F.binary_cross_entropy_with_logits(
                torch.cat([pos_s, neg_s]), 
                torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)]))
            loss.backward()
            opt.step()
            loss_sum += loss.item()
            n += 1
        pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

    # Evaluation
    model.eval()
    
    # MRR
    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_data), 200), desc="MRR", leave=False):
            batch = test_data.triples[i:i+200]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.base_model.score_tails(h, r)
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()

    # ECE
    pos = test_data.triples
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    confs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.base_model(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    ece, _ = expected_calibration_error(conf, labels)
    brier = brier_score(conf, labels)

    # AUROC
    ood_t = create_ood_dataset(train_data, test_data, "random", len(test_data))

    def get_unc(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
                _, var = model.predict_with_mc_samples(h, r, t, num_samples=20)
                uncs.append(var.cpu().numpy())
        return np.concatenate(uncs)

    auroc = compute_auroc(get_unc(test_data.triples), get_unc(ood_t))
    
    result = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "brier": brier, "auroc": auroc}
    print(f"Result: MRR={mrr:.4f}, H@1={h1:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    
    del model, base
    return result

In [ ]:
# Run multiple seeds
SEEDS = [42, 123, 456]
all_results = {}

for seed in SEEDS:
    all_results[seed] = train_and_evaluate(seed)

In [ ]:
# Aggregate results
metrics = ['mrr', 'hits@1', 'hits@10', 'ece', 'brier', 'auroc']

print("\n" + "="*70)
print("MCDROPOUT RESULTS (mean ± std)")
print("="*70)

summary = {}
for m in metrics:
    values = [all_results[s][m] for s in SEEDS]
    mean, std = np.mean(values), np.std(values)
    summary[m] = {"mean": mean, "std": std, "values": values}
    print(f"{m:>10}: {mean:.4f} ± {std:.4f}")

# Save
output = {"seeds": SEEDS, "results": all_results, "summary": summary}
with open('mcdropout_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("\nSaved to mcdropout_results.json")